In [118]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
from pickle import load, dump

import re
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Yavanash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [119]:
filepath = "../data/data.csv"
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

In [120]:
class Vocab():
    def __init__(self):
        self.itos = {0:PAD, 1:SOS, 2:EOS, 3:UNK}
        self.stoi = {PAD:0, SOS:1, EOS:2, UNK:3}
        self.count = 4

    def add(self, words):
        for word in words:
            if word not in self.stoi:
                self.stoi[word] = self.count
                self.itos[self.count] = word
                self.count += 1

In [121]:
def preprocess(text):
        text = re.sub(r'\W+', ' ', text)
        words = word_tokenize(text.lower())
        cleaned = [word for word in words if re.match(r"[a-z0-9+.,?!']", word)]
        return words

In [122]:
class NewsDataset(Dataset):
    def __init__(self, filepath, vocab, build_vocab=False):
        super().__init__()
        df = pd.read_csv(filepath)
        
        articles = df["article"].tolist()
        self.article = [preprocess(t) for t in articles]
        summary = df["text"].tolist()
        self.summary = [preprocess(t) for t in summary]

        if build_vocab:
            for x in self.article:
                vocab.add(x)
            for x in self.summary:
                vocab.add(x)

        self.vocab = vocab

    def __len__(self):
        return len(self.article)

    def __getitem__(self, idx):
        article = [self.vocab.stoi.get(word, self.vocab.stoi[UNK]) for word in self.article[idx]] #list of words in article
        summary = [self.vocab.stoi[SOS]] + [self.vocab.stoi.get(word, self.vocab.stoi[UNK]) for word in self.summary[idx]]#list of words in summary
        #add sos 

        if len(article) > 600:
            article = article[:600]

        if len(summary) > 100:
            summary = summary[:100]

        summary += [self.vocab.stoi[EOS]] 
        article_tensor = torch.tensor(article, dtype=torch.long)
        summary_tensor = torch.tensor(summary, dtype=torch.long)
        return article_tensor, summary_tensor

In [123]:
vocab = Vocab()
# data = NewsDataset(filepath, vocab, build_vocab=True)
data = NewsDataset(filepath, vocab, build_vocab=False)

In [126]:
VOCAB_SIZE = vocab.count
HIDDEN_SIZE = 128
VOCAB_SIZE

51947

In [67]:
# with open('../data/vocab.pkl', 'wb') as file:
#     dump(vocab, file)

In [125]:
with open('../data/vocab.pkl', 'rb') as file:
    vocab = load(file)

In [127]:
from torch.nn.utils.rnn import pad_sequence

#pads for entire batch at once
def padding(batch):
    article, summary = zip(*batch)
    article_batched = pad_sequence(article, batch_first=True, padding_value=0)
    summary_batched = pad_sequence(summary, batch_first=True, padding_value=0)

    return article_batched, summary_batched

In [128]:
train_loader = DataLoader(data, shuffle=True, batch_size=32, collate_fn=padding)
a,b = next(iter(train_loader))
a.shape, b.shape

(torch.Size([32, 600]), torch.Size([32, 67]))

In [129]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.dropout = nn.Dropout(0.1)
        self.bigru = nn.GRU(hidden_size, hidden_size, batch_first=True, bidirectional=True)
        self.out = nn.Linear(2*hidden_size, hidden_size)

    def forward(self, article):
        embed = self.dropout(self.embed(article))
        output, hidden = self.bigru(embed) #hidden = [2,bs,hidden]
        h = torch.cat((hidden[0], hidden[1]), dim=-1)
        hidden = self.out(h).unsqueeze(0)
        return output, hidden

In [130]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.wa = nn.Linear(hidden_size, hidden_size)
        self.va = nn.Linear(2*hidden_size, hidden_size)
        self.V = nn.Linear(hidden_size, 1)

    def forward(self, s_prev, keys):
        #query = decoder_prev , keys = encoder_all
        #s_prev = [num_layers, bs, hidden]
        query = s_prev.permute(1,0,2)
        scores = self.V(torch.tanh(self.wa(query) + self.va(keys)))
        alpha = torch.softmax(scores, dim=1)
        alpha = alpha.permute(0,2,1)

        context = torch.bmm(alpha, keys) # batch matrix multiplication
        return context, alpha

In [131]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(3*hidden_size, hidden_size, batch_first=True)
        self.attn = Attention(hidden_size)
        self.out = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        decoder_input = torch.empty(encoder_outputs.shape[0], 1, dtype=torch.long).fill_(1) # sos = 1
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attention_wts = []

        if target_tensor is not None:
            for i in range(target_tensor.shape[1]):
                decoder_output, decoder_hidden, attn_wts = self.forward_step(decoder_input, decoder_hidden, encoder_outputs)
                attention_wts.append(attn_wts)
                decoder_outputs.append(decoder_output)

                decoder_input = target_tensor[:,i].unsqueeze(-1)

        else:
            for i in range(MAXLEN):
                decoder_output, decoder_hidden, attn_wts = self.forward_step(decoder_input, decoder_hidden, encoder_outputs)
                attention_wts.append(attn_wts)
                decoder_outputs.append(decoder_output)

                topv,topi = decoder_output.topk(k=1,dim=-1) #dim=-1 -> vocab size , topi = [bs, 1, 1]
                decoder_input = topi.squeeze(-1).detach()

        outputs = torch.cat(decoder_outputs, dim=1)
        return outputs, decoder_hidden, attention_wts


    def forward_step(self, decoder_input, decoder_hidden, encoder_outputs):
        embed = self.dropout(self.embed(decoder_input))
        #s(t-1) = decoder_hidden
        #hn = encoder_outputs
        context, wts = self.attn(decoder_hidden, encoder_outputs)
        input = torch.cat((embed, context), dim=-1)
        # print(input.shape, decoder_hidden.shape)
        out, hidden = self.gru(input, decoder_hidden) #out = [bs, seqlen, hidden]
        out = self.out(out)
        return out, hidden, wts

In [132]:
e1 = Encoder(VOCAB_SIZE, HIDDEN_SIZE)
o,h=e1(a)
a.shape,o.shape, h.shape

(torch.Size([32, 600]), torch.Size([32, 600, 256]), torch.Size([1, 32, 128]))

In [133]:
d1 = Decoder(VOCAB_SIZE, HIDDEN_SIZE)
out, _, _ = d1(o,h,b)

In [135]:
out.shape

torch.Size([32, 67, 51947])

In [137]:
out = torch.argmax(out, dim=-1)
out.shape

torch.Size([32, 67])

In [ ]:
summary1 = [vocab.itos[i.item()] for i in out[0]]
summary1